# TiffWriter on the Ti2-E

Runs a short experiment with stimulation and a reference frame and
stores it with `TiffWriter`: one compressed TIFF file per frame and
data type. The same experiment is in `ome_zarr_writer.ipynb` and
`ome_zarr_writer_plate.ipynb`, so the three layouts can be compared.

## 1. Connect to the Ti2-E

`Ti2E()` loads `NikonTi2e.cfg`, applies the `System/Startup` preset and
selects the `Channel` group. Pass `config=r"..."` to load another file.
Close NIS-Elements and any other Micro-Manager first: a device can only be
opened by one program at a time.

In [ ]:
from faro_aoki.ti2e import Ti2E
import faro.core.utils as utils

mic = Ti2E()
utils.print_configs(mic.mmc)

## 2. Settings

Two imaging channels on every frame, blue light through the `GFP` preset on
the stim frames, and a reference channel on the last frame. The positions
start at the current stage position and step `STEP_UM` in x, so the sample
must reach that far. Positions have no `z`, so the focus is not moved.

In [ ]:
import os
from datetime import datetime

IMAGING_CHANNELS = [
    {"config": "iRFP", "exposure": 100},  # nuclear marker, segmented
    {"config": "RFP", "exposure": 100},
]
STIM_CHANNEL = {"config": "GFP", "exposure": 100}  # blue light, whole FOV
REF_CHANNEL = {"config": "YFP_Custom", "exposure": 200}  # last frame only
N_POSITIONS = 3
STEP_UM = 300.0  # x distance between positions
INTERVAL_S = 5.0
N_FRAMES = 6
STIM_FRAMES = [1, 3]

path = os.path.abspath(
    os.path.join("data", f"tiff-{datetime.now():%Y%m%d-%H%M%S}")
)  # data/ next to this notebook
print("results go to", path)

## 3. Set up the pipeline

Otsu segmentation on the first imaging channel, trackpy, `SimpleFE`, and
`StimWholeFOV`: the Ti2-E has no DMD, so stimulation lights the whole
field of view.

In [ ]:
from faro.core.data_structures import SegmentationMethod
from faro.core.pipeline import ImageProcessingPipeline
from faro.feature_extraction.simple import SimpleFE
from faro.segmentation.base import OtsuSegmentator
from faro.stimulation.base import StimWholeFOV
from faro.tracking.trackpy import TrackerTrackpy

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=OtsuSegmentator(),
        use_channel=0,
        save_tracked=True,
    )
]

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(),
    stimulator=StimWholeFOV(),
)

## 4. Create the TiffWriter

In [ ]:
from faro.core.writers import TiffWriter

writer = TiffWriter(storage_path=path)

## 5. Define the experiment

In [ ]:
from faro.core.data_structures import RTMSequence

x0, y0 = mic.mmc.getXYPosition()
positions = [{"x": x0 + i * STEP_UM, "y": y0} for i in range(N_POSITIONS)]

seq = RTMSequence(
    time_plan={"interval": INTERVAL_S, "loops": N_FRAMES},
    stage_positions=positions,
    channels=IMAGING_CHANNELS,
    stim_channels=[STIM_CHANNEL],
    stim_frames=STIM_FRAMES,
    ref_channels=[REF_CHANNEL],
    ref_frames=[-1],
)

events = list(seq)
print(f"{len(events)} events")

## 6. Run the experiment

The stage returns to the first position at the end, so you can run the
notebook again from the same place.

In [ ]:
from faro.core.controller import Controller

ctrl = Controller(mic, pipeline, writer=writer)
assert ctrl.validate_events(events), "fix the warnings above before running"
ctrl.run_experiment(events).wait()
ctrl.finish_experiment()
mic.mmc.setXYPosition(x0, y0)
print("Experiment complete.")

## 7. Inspect the TIFF folders

Every file is named `<position>_<timepoint>.tiff`, for example
`001_00003.tiff`. Expected file counts:

| Folder | Files | Content |
|--------|-------|---------|
| `raw/` | positions x frames | imaging channels, `(c, y, x)` |
| `stim/` | positions x frames | camera image taken during stimulation; an all-zero `uint8` placeholder on frames without stimulation |
| `ref/` | positions x 1 | reference channel of the last frame |
| `labels/`, `particles/` | positions x frames | segmentation and tracked labels |
| `stim_mask/` | positions x frames | stimulation masks |
| `tracks/` | one parquet per position | tracks and features |

The `stim/` placeholders have another shape and dtype than the real
readouts, so read the stim frames by name rather than stacking the
whole folder.

In [ ]:
for folder in sorted(os.listdir(path)):
    full = os.path.join(path, folder)
    if os.path.isdir(full):
        files = sorted(os.listdir(full))
        example = files[0] if files else "-"
        print(f"{folder + '/':14s} {len(files):4d} files, e.g. {example}")

In [ ]:
from glob import glob

import tifffile

for folder in ["raw", "stim", "ref", "labels", "particles", "stim_mask"]:
    files = sorted(glob(os.path.join(path, folder, "*.tiff")))
    if files:
        img = tifffile.imread(files[0])
        name = os.path.basename(files[0])
        print(f"{folder:10s} {name}: shape={img.shape}, dtype={img.dtype}")

In [ ]:
import matplotlib.pyplot as plt

fov, t_stim, t_last = 0, STIM_FRAMES[0], N_FRAMES - 1


def read(folder, t):
    return tifffile.imread(os.path.join(path, folder, f"{fov:03d}_{t:05d}.tiff"))


raw = read("raw", t_last)  # (c, y, x)
panels = [
    (raw[0], f"{IMAGING_CHANNELS[0]['config']} (last frame)"),
    (raw[1], f"{IMAGING_CHANNELS[1]['config']} (last frame)"),
    (read("labels", t_last), "Labels"),
    (read("stim", t_stim), f"Stim readout (frame {t_stim})"),
    (read("ref", t_last), f"Reference {REF_CHANNEL['config']}"),
]
fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
for ax, (img, title) in zip(axes, panels):
    img = img if img.ndim == 2 else img[0]
    ax.imshow(img, cmap="nipy_spectral" if title == "Labels" else "gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()

In [ ]:
import pandas as pd

for fov in range(N_POSITIONS):
    tracks = pd.read_parquet(os.path.join(path, "tracks", f"{fov}_latest.parquet"))
    print(
        f"Position {fov}: {tracks['particle'].nunique()} cells "
        f"across {tracks['timestep'].nunique()} frames"
    )
tracks.head()

`convert_tiff_to_omezarr.ipynb` turns this folder into an OME-Zarr
store. Its path:

In [ ]:
print(path)